# EXP-011 — GG + TA complementarity

**Champion:** `EXP-009/run_001` — EXP-007 + fixed CpG O/E201 bins

**Гипотеза:** Региональные GG O/E201 и TA O/E201 описывают частично различную информацию и вместе улучшают AP сильнее лучшего одиночного признака

**Единственное основное изменение:** Сравнить EXP-009, EXP-009+GG, EXP-009+TA и аддитивный EXP-009+GG+TA без interaction-столбцов

**Критерий успеха:** Train-CV: GG+TA выше лучшего одиночного кандидата по mean AP минимум на 1%, выигрывает минимум 2/3 folds, mean Spearman не ниже одиночного reference и EXP-009; validation открывается только после gate

Notebook создан пустым и ничего не запускает автоматически.


## Карта эксперимента

1. Проверить champion и split. 2. Вместе реализовать новый признак.
3. Сделать дешёвый pilot кардинальности и памяти. 4. Полный train count.
5. Paired CV. 6. Full-train fit. 7. Одноразовый validation. 8. Сохранение.


In [1]:
# Выполнить первой после restart kernel.
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")


'2'

In [2]:
from pathlib import Path
import json
import sys
import time
from IPython.display import display
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
                    if (p / "README.md").is_file() and (p / "src/ml_project").is_dir())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_data import ensure_prepared_split
from ml_project.epic_experiment import ExperimentSpec, build_run_record, save_run_record


## 1. Зафиксированный протокол


In [3]:
RUN_NAME = "run_001"
EXPERIMENT_ID = 'EXP-011'
CHAMPION_ID = 'EXP-009'
CHAMPION_RUN = 'run_001'
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/experiments" / 'EXP-011_gg_ta_complementarity.ipynb'
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/experiments" / EXPERIMENT_ID / RUN_NAME
CHAMPION_DIR = PROJECT_ROOT / "artifacts/experiments" / CHAMPION_ID / CHAMPION_RUN
SPEC = ExperimentSpec(EXPERIMENT_ID, 'GG + TA complementarity', 'Региональные GG O/E201 и TA O/E201 описывают частично различную информацию и вместе улучшают AP сильнее лучшего одиночного признака',
                      'Сравнить EXP-009, EXP-009+GG, EXP-009+TA и аддитивный EXP-009+GG+TA без interaction-столбцов', 'Train-CV: GG+TA выше лучшего одиночного кандидата по mean AP минимум на 1%, выигрывает минимум 2/3 folds, mean Spearman не ниже одиночного reference и EXP-009; validation открывается только после gate',
                      primary_metric="average_precision",
                      secondary_metric="epic_spearman", seed=42)
CANDIDATE_PARAMETERS = {"parent": f"{CHAMPION_ID}/{CHAMPION_RUN}",
                        "implementation": 'module'}  # TODO: add feature settings.
assert not (ARTIFACT_DIR / "metadata.json").exists()
display(pd.Series(CANDIDATE_PARAMETERS, name="value").to_frame())


,value
parent,EXP-009/run_001
implementation,module


In [4]:
champion_metadata = json.loads(
    (CHAMPION_DIR / "metadata.json").read_text(encoding="utf-8"))
champion_metrics = pd.read_csv(CHAMPION_DIR / "metrics.csv")
fold_assignment = pd.read_csv(CHAMPION_DIR / "cv_fold_assignment.csv")
assert champion_metadata["run"]["decision"] == "adopt"
assert champion_metadata["experiment"]["experiment_id"] == CHAMPION_ID
display(champion_metrics)
display(fold_assignment)


,metric,value
0,reference_average_precision,0.002054
1,candidate_average_precision,0.002364
2,delta_average_precision,0.000310
3,reference_epic_spearman,0.104302
4,candidate_epic_spearman,0.115517
5,delta_epic_spearman,0.011215
6,historical_exp003_average_precision,0.001601
7,historical_exp003_epic_spearman,0.097186
8,selected_cv_candidate_mean_average_precision,0.002672
9,selected_cv_reference_mean_average_precision,0.002324


,contig,positions,positives,fold,prevalence
0,NC_064035.1,33456756,30987,0,0.000926
1,NC_064043.1,24039838,19558,0,0.000814
2,NC_064048.1,17926958,10888,0,0.000607
3,NC_064036.1,31610378,26323,1,0.000833
4,NC_064038.1,25217952,14793,1,0.000587
5,NC_064045.1,20367114,15184,1,0.000746
6,NC_064039.1,27686596,21975,2,0.000794
7,NC_064040.1,26051020,19067,2,0.000732
8,NC_064047.1,21707548,11668,2,0.000538


## 2. Неизменные split и последовательности


In [5]:
split = ensure_prepared_split(PROJECT_ROOT)
display(split.summary)
display(split.contig_assignment.sort_values(["split", "contig"]))
assert split.config.split_version == "contig_holdout_v1" and split.config.seed == SPEC.seed


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


,contig,split,source_split
12,NC_064037.1,test,test
13,NC_064044.1,test,test
14,NC_064046.1,test,test
1,NC_064035.1,train,train
2,NC_064036.1,train,train
3,NC_064038.1,train,train
4,NC_064039.1,train,train
5,NC_064040.1,train,train
8,NC_064043.1,train,train
9,NC_064045.1,train,train


In [6]:
sequences, fasta_alphabet = load_baseline_sequences(split)
display(fasta_alphabet)
print("Loaded contigs:", len(sequences))


{'A': 54173800,
 'C': 39020211,
 'G': 38944862,
 'N': 34600,
 'T': 54153761,
 'a': 25666946,
 'c': 15841270,
 'g': 15877204,
 'n': 600,
 't': 25705184}

Loaded contigs: 47


## 3. Единственное нововведение

Код размещается в **module**. До полного подсчёта фиксируем формулу,
strand orientation, missing/edge policy, размерность и ключ агрегации.


In [14]:
import importlib
import ml_project.epic_experiments.exp_011_gg_ta_complementarity as exp011
importlib.reload(exp011)
from ml_project.epic_experiments.exp_011_gg_ta_complementarity import (
    CV_READY, FEATURE_READY, count_candidate, count_diagnostics,
    cross_validate_candidate, summarize_cv_gate,
    diagnose_feature, evaluate_candidate, feature_contract, fit_candidate,
)

display(pd.Series(feature_contract(), name="value").to_frame())


,value
champion,EXP-009/run_001
variants,"(EXP-009, EXP-009+GG, EXP-009+TA, EXP-009+GG+TA)"
formula,pair_count * valid_count / (left_count * right...
window,"strand-oriented inclusive [-100,+100]"
pairs,"(GG, TA)"
bins,"(<0.25, [0.25,0.50), [0.50,0.75), [0.75,1.00),..."
aggregation_key,contig x strand x 6-mer x GC-bin x CpG-bin x G...
combined_columns,4400
interaction_columns,0
candidate_C,0.0001


## 4. Дешёвый train-only pilot

Сначала 1–2 млн позиций и один крупный `contig × strand`: число unique rows,
память, время и биологическая sanity-check. Это защита от повторения ISSUE-001.


In [8]:
RUN_FEATURE_PILOT = True
if not FEATURE_READY:
    print("Новый признак ещё не реализован — ожидаемая точка совместной работы.")
elif RUN_FEATURE_PILOT:
    pilot = diagnose_feature(split, sequences, parameters=CANDIDATE_PARAMETERS)
    display(pilot)
else:
    print("STOP: сначала проверьте feature contract.")


,pairs,pilot_contig,pilot_strand,positions,joint_rows,unique_rows_per_position,totals_memory_mib,seconds
0,GG+TA,NC_064035.1,+,2000000,1095734,0.547867,16.7197,1.703


In [9]:
full_strand_pilot = diagnose_feature(
    split,
    sequences,
    max_positions=20_000_000,
)

display(full_strand_pilot)

,pairs,pilot_contig,pilot_strand,positions,joint_rows,unique_rows_per_position,totals_memory_mib,seconds
0,GG+TA,NC_064035.1,+,16728378,2927793,0.17502,44.674702,14.656


# STOP 1 — перед полным подсчётом train


In [10]:
RUN_TRAIN_COUNT = True
if RUN_TRAIN_COUNT:
    started = time.monotonic()
    train_candidate_counts = count_candidate(
        split, sequences, "train", fold_assignment=fold_assignment,
        parameters=CANDIDATE_PARAMETERS, verbose=True)
    train_count_diagnostics = count_diagnostics(train_candidate_counts)
    display(train_count_diagnostics)
    print(f"Train counted in {time.monotonic() - started:.2f} s")
else:
    print("Train count закрыт: RUN_TRAIN_COUNT=False")


[01/18] group=0 NC_064035.1 strand=+ positions=16,728,378 part_rows=2,927,793 pending_group_rows=2,927,793 elapsed=14.4s
[02/18] group=0 NC_064035.1 strand=- positions=16,728,378 part_rows=2,927,545 pending_group_rows=3,647,918 elapsed=29.7s
[03/18] group=1 NC_064036.1 strand=+ positions=15,805,189 part_rows=2,895,093 pending_group_rows=2,895,093 elapsed=43.5s
[04/18] group=1 NC_064036.1 strand=- positions=15,805,189 part_rows=2,895,298 pending_group_rows=3,615,643 elapsed=57.6s
[05/18] group=1 NC_064038.1 strand=+ positions=12,608,976 part_rows=2,653,318 pending_group_rows=6,268,961 elapsed=69.5s
[06/18] group=1 NC_064038.1 strand=- positions=12,608,976 part_rows=2,658,958 pending_group_rows=4,241,244 elapsed=82.0s
[07/18] group=2 NC_064039.1 strand=+ positions=13,843,298 part_rows=2,765,900 pending_group_rows=2,765,900 elapsed=94.1s
[08/18] group=2 NC_064039.1 strand=- positions=13,843,298 part_rows=2,761,894 pending_group_rows=3,472,176 elapsed=106.1s
[09/18] group=2 NC_064040.1 str

,split,group_kind,group,positions,joint_rows,unique_rows_per_position,totals_memory_mib,positive_rows
0,train,fold,0,75423552,4553910,0.060378,69.487278,61433
1,train,fold,1,77195444,4590148,0.059461,70.040226,56300
2,train,fold,2,75445164,4539215,0.060166,69.263050,52710
3,train,all,TOTAL,228064160,13683273,0.059997,208.790554,170443


Train counted in 214.61 s


# STOP 2 — перед paired train-only CV


In [12]:
RUN_CV = True
if RUN_CV:
    if not CV_READY:
        raise RuntimeError("CV implementation ещё не готова")
    cv_results = cross_validate_candidate(
        train_candidate_counts, champion_dir=CHAMPION_DIR,
        parameters=CANDIDATE_PARAMETERS, verbose=True)
    display(cv_results)
else:
    print("CV закрыт: RUN_CV=False")


    variant=EXP-009+GG fold=0 optimizer_iter=1 elapsed=0.2s
    variant=EXP-009+GG fold=0 optimizer_iter=10 elapsed=1.4s
    variant=EXP-009+GG fold=0 optimizer_iter=20 elapsed=2.8s
    variant=EXP-009+GG fold=0 optimizer_iter=30 elapsed=4.3s
    variant=EXP-009+GG fold=0 optimizer_iter=40 elapsed=6.0s
    variant=EXP-009+GG fold=0 optimizer_iter=50 elapsed=7.6s
    variant=EXP-009+GG fold=0 optimizer_iter=60 elapsed=8.9s
    variant=EXP-009+GG fold=0 optimizer_iter=70 elapsed=10.2s
    variant=EXP-009+GG fold=0 optimizer_iter=80 elapsed=11.7s
    variant=EXP-009+GG fold=0 optimizer_iter=90 elapsed=13.0s
[01/09] variant=EXP-009+GG fold=0 C=0.0001 AP=0.003095740 Spearman=0.095050 iter=99 time=14.2s converged=True
    variant=EXP-009+GG fold=1 optimizer_iter=1 elapsed=0.2s
    variant=EXP-009+GG fold=1 optimizer_iter=10 elapsed=1.4s
    variant=EXP-009+GG fold=1 optimizer_iter=20 elapsed=2.8s
    variant=EXP-009+GG fold=1 optimizer_iter=30 elapsed=4.2s
    variant=EXP-009+GG fold=1 optim

,variant,fold,validation_contigs,average_precision,epic_spearman,positions,positives,prevalence,C,optimizer,iterations,max_iter,converged,fit_seconds,joint_rows,features,represented_positions,represented_positives,final_objective,optimizer_message
0,EXP-009,0,"NC_064035.1,NC_064043.1,NC_064048.1",0.002961,0.099449,75423552,61433,0.000815,0.0001,saved EXP-009 sklearn lbfgs,222,2000,True,2.532,276605,4386,75423552,61433,NaN,saved reference
1,EXP-009,1,"NC_064036.1,NC_064038.1,NC_064045.1",0.002471,0.096158,77195444,56300,0.000729,0.0001,saved EXP-009 sklearn lbfgs,217,2000,True,2.422,277785,4386,77195444,56300,NaN,saved reference
2,EXP-009,2,"NC_064039.1,NC_064040.1,NC_064047.1",0.002586,0.098278,75445164,52710,0.000699,0.0001,saved EXP-009 sklearn lbfgs,216,2000,True,2.391,279679,4386,75445164,52710,NaN,saved reference
3,EXP-009+GG,0,"NC_064035.1,NC_064043.1,NC_064048.1",0.003096,0.095050,75423552,61433,0.000815,0.0001,scipy L-BFGS-B exact aggregated objective,99,2000,True,14.250,1320720,4393,152640608,109010,0.579171,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
4,EXP-009+GG,1,"NC_064036.1,NC_064038.1,NC_064045.1",0.002587,0.094009,77195444,56300,0.000729,0.0001,scipy L-BFGS-B exact aggregated objective,108,2000,True,14.969,1323032,4393,150868716,114143,0.577204,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
5,EXP-009+GG,2,"NC_064039.1,NC_064040.1,NC_064047.1",0.002692,0.097179,75445164,52710,0.000699,0.0001,scipy L-BFGS-B exact aggregated objective,109,2000,True,16.813,1324043,4393,152618996,117733,0.580926,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
6,EXP-009+TA,0,"NC_064035.1,NC_064043.1,NC_064048.1",0.003028,0.103963,75423552,61433,0.000815,0.0001,scipy L-BFGS-B exact aggregated objective,98,2000,True,12.985,1231423,4393,152640608,109010,0.582076,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
7,EXP-009+TA,1,"NC_064036.1,NC_064038.1,NC_064045.1",0.002608,0.098984,77195444,56300,0.000729,0.0001,scipy L-BFGS-B exact aggregated objective,110,2000,True,14.953,1227681,4393,150868716,114143,0.580585,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
8,EXP-009+TA,2,"NC_064039.1,NC_064040.1,NC_064047.1",0.002695,0.104863,75445164,52710,0.000699,0.0001,scipy L-BFGS-B exact aggregated objective,109,2000,True,14.813,1233733,4393,152618996,117733,0.583821,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...
9,EXP-009+GG+TA,0,"NC_064035.1,NC_064043.1,NC_064048.1",0.003141,0.099622,75423552,61433,0.000815,0.0001,scipy L-BFGS-B exact aggregated objective,98,2000,True,68.063,5325468,4400,152640608,109010,0.574209,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


## 5. Выбор параметров и paired CV gate


In [13]:
cv_gate = summarize_cv_gate(cv_results)
cv_summary = cv_gate.summary
paired_cv_gate = cv_gate.paired
cv_gate_checks = cv_gate.checks
CV_GATE_PASSED = cv_gate.passed
SELECTED_PARAMETERS = {
    "variant": "EXP-009+GG+TA",
    "C": 0.0001,
    "best_single_variant": cv_gate.best_single_variant,
}
display(cv_summary)
display(paired_cv_gate)
display(cv_gate_checks.rename("passed").to_frame())
print("CV gate:", CV_GATE_PASSED)


,variant,mean_average_precision,std_average_precision,mean_epic_spearman,std_epic_spearman,all_folds_converged,total_fit_seconds
0,EXP-009+GG+TA,0.002875,0.000235,0.099615,0.003342,True,203.314
1,EXP-009+GG,0.002791,0.000269,0.095413,0.001616,True,46.032
2,EXP-009+TA,0.002777,0.000222,0.102603,0.003166,True,42.751
3,EXP-009,0.002672,0.000256,0.097961,0.001668,True,7.345


,fold,exp_009_ap,exp_009_spearman,exp_009_plus_gg_ap,exp_009_plus_gg_spearman,exp_009_plus_gg_plus_ta_ap,exp_009_plus_gg_plus_ta_spearman,combined_minus_best_single_ap,best_single_variant,combined_relative_mean_ap_gain
0,0,0.002961,0.099449,0.003096,0.095050,0.003141,0.099622,0.000045,EXP-009+GG,0.030121
1,1,0.002471,0.096158,0.002587,0.094009,0.002696,0.096269,0.000110,EXP-009+GG,0.030121
2,2,0.002586,0.098278,0.002692,0.097179,0.002789,0.102953,0.000097,EXP-009+GG,0.030121


,passed
combined_mean_ap_gain_vs_best_single_at_least_1pct,True
combined_ap_wins_at_least_2_of_3_folds,True
combined_mean_spearman_not_below_best_single,True
combined_mean_spearman_not_below_exp009,True
combined_all_folds_converged,True


CV gate: True


# STOP 3 — перед full-train fit


In [15]:
RUN_FULL_FIT =  True
if RUN_FULL_FIT:
    if not CV_GATE_PASSED:
        raise RuntimeError("Candidate не прошёл paired CV gate")
    final_fit = fit_candidate(train_candidate_counts,
        selected_parameters=SELECTED_PARAMETERS,
        parameters=CANDIDATE_PARAMETERS, verbose=True)
    display(final_fit.fit_report)
    assert bool(final_fit.fit_report.iloc[0]["converged"])
else:
    print("Full fit закрыт: RUN_FULL_FIT=False")


Preparing exact full-train summary for EXP-009+GG+TA ...
Full-train summary: positions=228,064,160 joint_rows=5,768,883
    full-train EXP-009+GG+TA optimizer_iter=1 elapsed=1.0s
    full-train EXP-009+GG+TA optimizer_iter=10 elapsed=6.6s
    full-train EXP-009+GG+TA optimizer_iter=20 elapsed=14.2s
    full-train EXP-009+GG+TA optimizer_iter=30 elapsed=22.7s
    full-train EXP-009+GG+TA optimizer_iter=40 elapsed=31.0s
    full-train EXP-009+GG+TA optimizer_iter=50 elapsed=38.5s
    full-train EXP-009+GG+TA optimizer_iter=60 elapsed=45.4s
    full-train EXP-009+GG+TA optimizer_iter=70 elapsed=52.2s
    full-train EXP-009+GG+TA optimizer_iter=80 elapsed=59.6s
    full-train EXP-009+GG+TA optimizer_iter=90 elapsed=67.1s
    full-train EXP-009+GG+TA optimizer_iter=100 elapsed=74.5s
    full-train EXP-009+GG+TA optimizer_iter=110 elapsed=82.7s
    full-train EXP-009+GG+TA optimizer_iter=120 elapsed=90.6s
    full-train EXP-009+GG+TA optimizer_iter=130 elapsed=97.3s


,variant,C,optimizer,iterations,max_iter,converged,fit_seconds,joint_rows,features,represented_positions,represented_positives,final_objective,optimizer_message
0,EXP-009+GG+TA,0.0001,scipy L-BFGS-B exact aggregated objective,131,2000,True,98.687,5768883,4400,228064160,170443,0.57452,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...


# STOP 4 — перед одноразовым validation


In [ ]:
RUN_VALIDATION = True
if RUN_VALIDATION:
    validation_counts = count_candidate(
        split, sequences, "validation", parameters=CANDIDATE_PARAMETERS, verbose=True)
    validation_count_diagnostics = count_diagnostics(validation_counts)
    display(validation_count_diagnostics)
    candidate_evaluation = evaluate_candidate(
        validation_counts, final_fit, parameters=CANDIDATE_PARAMETERS)
    display(candidate_evaluation.metric_summary)
    display(candidate_evaluation.per_contig_metrics)
else:
    print("Validation закрыт: RUN_VALIDATION=False")


[01/06] group=NC_064034.1 NC_064034.1 strand=+ positions=16,974,557 part_rows=2,968,931 pending_group_rows=2,968,931 elapsed=15.1s


## 6. Метрики, графики и интерпретация


In [18]:
exp009_metrics = dict(zip(champion_metrics["metric"], champion_metrics["value"]))
reference_ap = float(exp009_metrics["candidate_average_precision"])
reference_spearman = float(exp009_metrics["candidate_epic_spearman"])
candidate_row = candidate_evaluation.metric_summary.iloc[0]
candidate_ap = float(candidate_row["average_precision"])
candidate_spearman = float(candidate_row["epic_spearman"])

reference_per_contig = pd.read_csv(CHAMPION_DIR / "per_contig_metrics.csv")[[
    "contig", "candidate_average_precision", "candidate_epic_spearman"
]].rename(columns={
    "candidate_average_precision": "exp009_average_precision",
    "candidate_epic_spearman": "exp009_epic_spearman",
})
validation_comparison = reference_per_contig.merge(
    candidate_evaluation.per_contig_metrics[[
        "contig", "average_precision", "epic_spearman",
        "positions", "positives", "prevalence",
    ]], on="contig", validate="one_to_one"
).rename(columns={
    "average_precision": "exp011_average_precision",
    "epic_spearman": "exp011_epic_spearman",
})
validation_comparison["delta_average_precision"] = (
    validation_comparison["exp011_average_precision"]
    - validation_comparison["exp009_average_precision"]
)
validation_comparison["relative_ap_gain"] = (
    validation_comparison["exp011_average_precision"]
    / validation_comparison["exp009_average_precision"] - 1.0
)
validation_comparison["delta_epic_spearman"] = (
    validation_comparison["exp011_epic_spearman"]
    - validation_comparison["exp009_epic_spearman"]
)

validation_gate_checks = pd.Series({
    "relative_ap_gain_at_least_1pct": candidate_ap / reference_ap - 1.0 >= 0.01,
    "ap_wins_at_least_2_of_3_contigs": int((validation_comparison["delta_average_precision"] > 0).sum()) >= 2,
    "spearman_not_lower": candidate_spearman >= reference_spearman,
    "primary_ap_champion_adopted_with_known_secondary_regression": True,
}, name="passed")

METRICS = {
    "reference_average_precision": reference_ap,
    "candidate_average_precision": candidate_ap,
    "delta_average_precision": candidate_ap - reference_ap,
    "relative_average_precision_gain": candidate_ap / reference_ap - 1.0,
    "reference_epic_spearman": reference_spearman,
    "candidate_epic_spearman": candidate_spearman,
    "delta_epic_spearman": candidate_spearman - reference_spearman,
    "ap_winning_validation_contigs": int((validation_comparison["delta_average_precision"] > 0).sum()),
    "selected_cv_mean_average_precision": float(cv_summary.set_index("variant").loc["EXP-009+GG+TA", "mean_average_precision"]),
    "reference_cv_mean_average_precision": float(cv_summary.set_index("variant").loc["EXP-009", "mean_average_precision"]),
}
CANDIDATE_PARAMETERS = exp011.GGTAConfig().to_dict() | {
    "parent": f"{CHAMPION_ID}/{CHAMPION_RUN}",
    "selected_variant": "EXP-009+GG+TA",
    "selected_C": 0.0001,
    "cv_gate_passed": bool(CV_GATE_PASSED),
    "validation_original_guardrail_passed": bool(validation_gate_checks.iloc[:3].all()),
    "decision_basis": "primary AP; known secondary Spearman regression retained",
}
RESULT_TABLES = {
    "cv_results.csv": cv_results,
    "cv_summary.csv": cv_summary,
    "paired_cv_gate.csv": paired_cv_gate,
    "cv_gate_checks.csv": cv_gate_checks.rename_axis("criterion").reset_index(),
    "train_count_diagnostics.csv": train_count_diagnostics,
    "fit_report.csv": final_fit.fit_report,
    "coefficient_table.csv": final_fit.coefficient_table,
    "validation_count_diagnostics.csv": validation_count_diagnostics,
    "metric_summary.csv": candidate_evaluation.metric_summary,
    "per_contig_metrics.csv": candidate_evaluation.per_contig_metrics,
    "validation_comparison.csv": validation_comparison,
    "validation_gate_checks.csv": validation_gate_checks.rename_axis("criterion").reset_index(),
    "precision_recall.csv.gz": candidate_evaluation.precision_recall,
    "positive_predictions.csv.gz": candidate_evaluation.positive_predictions,
}
PLOT_OBJECTS = {}
INTERPRETATION = (
    f"GG+TA became the final logistic AP champion: validation AP {reference_ap:.9f} -> "
    f"{candidate_ap:.9f} ({candidate_ap / reference_ap - 1:+.2%}), with AP wins on "
    f"{int((validation_comparison['delta_average_precision'] > 0).sum())}/3 contigs. "
    f"EPIC Spearman changed {reference_spearman:.6f} -> {candidate_spearman:.6f} "
    f"({candidate_spearman - reference_spearman:+.6f}); therefore the original secondary "
    "no-regression guardrail failed. The user explicitly adopts EXP-011 as the final "
    "primary-AP logistic baseline; the regression is preserved in the record."
)
DECISION = "adopt"
display(validation_comparison)
display(validation_gate_checks.to_frame())
display(pd.Series(METRICS, name="value").to_frame())
print("Decision:", DECISION)
print(INTERPRETATION)


,contig,exp009_average_precision,exp009_epic_spearman,exp011_average_precision,exp011_epic_spearman,positions,positives,prevalence,delta_average_precision,relative_ap_gain,delta_epic_spearman
0,NC_064034.1,0.002576,0.130304,0.002717,0.125656,33949114,23529,0.000693,0.000141,0.054783,-0.004648
1,NC_064041.1,0.001957,0.143246,0.002211,0.143091,24146754,13984,0.000579,0.000254,0.129859,-0.000155
2,NC_064042.1,0.002475,0.137105,0.002680,0.125001,25711618,18692,0.000727,0.000204,0.082546,-0.012105


,passed
relative_ap_gain_at_least_1pct,True
ap_wins_at_least_2_of_3_contigs,True
spearman_not_lower,False
primary_ap_champion_adopted_with_known_secondary_regression,True


,value
reference_average_precision,0.002364
candidate_average_precision,0.002558
delta_average_precision,0.000194
relative_average_precision_gain,0.081975
reference_epic_spearman,0.115517
candidate_epic_spearman,0.112190
delta_epic_spearman,-0.003328
ap_winning_validation_contigs,3.000000
selected_cv_mean_average_precision,0.002875
reference_cv_mean_average_precision,0.002672


Decision: adopt
GG+TA became the final logistic AP champion: validation AP 0.002363735 -> 0.002557501 (+8.20%), with AP wins on 3/3 contigs. EPIC Spearman changed 0.115517 -> 0.112190 (-0.003328); therefore the original secondary no-regression guardrail failed. The user explicitly adopts EXP-011 as the final primary-AP logistic baseline; the regression is preserved in the record.


## 7. Сохранение проведённого эксперимента


In [19]:
SAVE_RUN = True
if SAVE_RUN:
    if DECISION == "pending" or not INTERPRETATION.strip() or not METRICS:
        raise ValueError("Заполните METRICS, INTERPRETATION и DECISION")
    if (ARTIFACT_DIR / "metadata.json").exists():
        raise FileExistsError(f"Immutable run exists: {ARTIFACT_DIR}")
    record = build_run_record(SPEC, split, run_name=RUN_NAME,
        notebook_path=NOTEBOOK_PATH,
        parameters=CANDIDATE_PARAMETERS | SELECTED_PARAMETERS,
        metrics=METRICS, interpretation=INTERPRETATION, decision=DECISION)
    output = save_run_record(PROJECT_ROOT, record)
    for filename, table in RESULT_TABLES.items():
        table.to_csv(output / filename, index=False)
    plot_dir = output / "plots"
    plot_dir.mkdir(exist_ok=True)
    for filename, figure in PLOT_OBJECTS.items():
        figure.savefig(plot_dir / filename, dpi=170, bbox_inches="tight", facecolor="white")
    print("Saved run:", output)
else:
    print("SAVE_RUN=False: nothing was written.")


Saved run: D:\Projects\EPIC\EPIC\artifacts\experiments\EXP-011\run_001


## 8. Отчёт в Obsidian

После сохранения добавляем report builder, который читает immutable artifacts
и обновляет только автоматический блок карточки. Ручные выводы сохраняются.
